# Avaliação spaCy para âncora de entidades

Compara spaCy `pt_core_news_lg` com a heurística atual. Mede somente a salvaguarda de entidade, não retrieval nem o veredito do LLM. GPU não é necessária.

In [ ]:
%pip -q install spacy
!python -m spacy download pt_core_news_lg -q


In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded))
print(input_path)


In [ ]:
import json, re, time, unicodedata
import spacy
nlp = spacy.load('pt_core_news_lg', disable=['parser', 'lemmatizer'])
ALIASES = {'tse': {'tribunal superior eleitoral'}, 'inss': {'instituto nacional do seguro social'}, 'anvisa': {'agencia nacional de vigilancia sanitaria'}}
PARTICLES, ARTICLES = {'da','das','de','do','dos','e'}, {'a','o','os','as','um','uma'}
WORD = re.compile(r'[A-Za-zÀ-ÖØ-öø-ÿ]+'); ACRONYM = re.compile(r'^[A-ZÀ-Ý]{2,6}$')
def norm(value):
    return ' '.join(''.join(c for c in unicodedata.normalize('NFKD', value) if not unicodedata.combining(c)).casefold().split())
def heuristic(claim):
    tokens = WORD.findall(claim); entities=[]; i=0
    while i < len(tokens):
        t=tokens[i]
        if ACRONYM.match(t) and t not in {'OK','AI','TV'}: entities.append(t); i+=1; continue
        if not t[:1].isupper() or t.lower() in ARTICLES: i+=1; continue
        parts=[t]; capitals=1; j=i+1
        while j < len(tokens) and (tokens[j][:1].isupper() or (tokens[j].lower() in PARTICLES and j+1 < len(tokens) and tokens[j+1][:1].isupper())):
            parts.append(tokens[j]); capitals += int(tokens[j][:1].isupper()); j+=1
        if capitals >= 2: entities.append(' '.join(parts))
        i=max(j,i+1)
    return sorted({norm(e) for e in entities}, key=len, reverse=True)
def spacy_ents(claim):
    return sorted({norm(e.text) for e in nlp(claim).ents if e.label_ in {'PER','ORG','LOC','MISC','GPE'}}, key=len, reverse=True)
def present(entities, text):
    text=norm(text)
    def variants(entity): return {entity} | ALIASES.get(entity,set())
    return all(any(re.search(rf'(?<!\w){re.escape(v)}(?!\w)', text) for v in variants(e)) for e in entities)
rows=[json.loads(line) for line in open(input_path, encoding='utf-8') if line.strip()]


In [ ]:
out=[]; elapsed={'heuristic':0., 'spacy':0.}
for row in rows:
    start=time.perf_counter(); h=heuristic(row['claim']); ha=present(h,row['evidence']); elapsed['heuristic'] += time.perf_counter()-start
    start=time.perf_counter(); s=spacy_ents(row['claim']); sa=present(sorted(set(s+h),key=len,reverse=True),row['evidence']); elapsed['spacy'] += time.perf_counter()-start
    out.append({**row,'heuristic_entities':h,'spacy_entities':s,'heuristic_anchor':ha,'spacy_anchor':sa})
def metric(key):
    return {'accuracy':sum(r[key]==r['expected_anchor'] for r in out)/len(out), 'false_positive':sum(not r['expected_anchor'] and r[key] for r in out), 'false_negative':sum(r['expected_anchor'] and not r[key] for r in out), 'mean_latency_ms':elapsed[key.replace('_anchor','')]*1000/len(out)}
report={'n':len(out),'heuristic':metric('heuristic_anchor'),'spacy':metric('spacy_anchor'),'cases':out,'scope':'Entity-anchor safeguard only; not retrieval or LLM verdict.'}
open('ner_anchor_spacy_v1.json','w',encoding='utf-8').write(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in report.items() if k!='cases'},ensure_ascii=False,indent=2))
files.download('ner_anchor_spacy_v1.json')
